# 02 Pipeline (Full Read Pipeline Template)

This is the **full-read pipeline template**. Clone Read and Write blocks as needed for many-to-many pipelines: every governed source is read in full, while each target can still use its governed write strategy such as overwrite, append, SCD1, or SCD2.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.1.0 | Voyce | 13 Jul 2026 |
| v0.2.0 | Voyce | 16 Sep 2026 |


# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_guardrail_coverage,
    check_schema,
    check_sensitive_data,
    check_source_drift,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
)

# 1. Data Contract

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts.

In [ ]:
CONTRACTS = widget_select_data_contract(spark_session=spark)

# 2. Full Read

Each cloneable Read block reads one complete governed source. Add or remove Read blocks as needed; this template does not perform source-side incremental reads.

In [ ]:
# Dictionary used to keep multiple source reads for later steps.
# Key = READ_NAME (for example "orders").
# Value = the full pipeline_read() result, including the source DataFrame and table_id.
sources = {}

## READ 1 — Orders

In [ ]:
READ_NAME = "orders"
READ_STORE = "Bronze"
READ_SCHEMA = "demo"
READ_TABLE = "orders"
READ_QUERY = None

# Read the whole source table and get its FabricOps table_id.
source = pipeline_read(store=READ_STORE, schema=READ_SCHEMA, table_name=READ_TABLE, query=READ_QUERY)

df = source["dataframe"]
table_id = source["table_id"]

# Stop here if this source is older than the allowed Freshness rule.
freshness_result = check_freshness(table_id, raise_on_failure=True)

# Stop here if the source columns or data types no longer match the contract.
schema_result = check_schema(df, table_id=table_id, raise_on_failure=True)

# Run the Data Quality rules. Keep the checked DataFrame and failed rows for optional use.
dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
dq_df = dq_result.get("dataframe", df)
dq_failed_values = dq_result.get("failed_values")

# Refresh the saved profile for the complete source table.
profile_result = profile_table(table_id=table_id)

# Add this read result to the sources dictionary.
# Later cells can reuse its DataFrame and table_id by READ_NAME.
sources[READ_NAME] = source

# Optional development inspection — uncomment only when needed.
# display(df)
# display(profile_result["profile"])
# display(dq_df)
# display(dq_failed_values)

## READ 2 — Products

In [ ]:
READ_NAME = "products"
READ_STORE = "Bronze"
READ_SCHEMA = "demo"
READ_TABLE = "products"
READ_QUERY = None

# Read the whole source table and get its FabricOps table_id.
source = pipeline_read(store=READ_STORE, schema=READ_SCHEMA, table_name=READ_TABLE, query=READ_QUERY)

df = source["dataframe"]
table_id = source["table_id"]

# Stop here if this source is older than the allowed Freshness rule.
freshness_result = check_freshness(table_id, raise_on_failure=True)

# Stop here if the source columns or data types no longer match the contract.
schema_result = check_schema(df, table_id=table_id, raise_on_failure=True)

# Run the Data Quality rules. Keep the checked DataFrame and failed rows for optional use.
dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
dq_df = dq_result.get("dataframe", df)
dq_failed_values = dq_result.get("failed_values")

# Refresh the saved profile for the complete source table.
profile_result = profile_table(table_id=table_id)

# Add this read result to the sources dictionary.
# Later cells can reuse its DataFrame and table_id by READ_NAME.
sources[READ_NAME] = source

# Optional development inspection — uncomment only when needed.
# display(df)
# display(profile_result["profile"])
# display(dq_df)
# display(dq_failed_values)

## READ 3 — Order History

In [ ]:
READ_NAME = "history"
READ_STORE = "Gold"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"
READ_QUERY = None

# Read the whole source table and get its FabricOps table_id.
source = pipeline_read(store=READ_STORE, schema=READ_SCHEMA, table_name=READ_TABLE, query=READ_QUERY)

df = source["dataframe"]
table_id = source["table_id"]

# Stop here if this source is older than the allowed Freshness rule.
freshness_result = check_freshness(table_id, raise_on_failure=True)

# Stop here if the source columns or data types no longer match the contract.
schema_result = check_schema(df, table_id=table_id, raise_on_failure=True)

# Run the Data Quality rules. Keep the checked DataFrame and failed rows for optional use.
dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
dq_df = dq_result.get("dataframe", df)
dq_failed_values = dq_result.get("failed_values")

# Refresh the saved profile for the complete source table.
profile_result = profile_table(table_id=table_id)

# Add this read result to the sources dictionary.
# Later cells can reuse its DataFrame and table_id by READ_NAME.
sources[READ_NAME] = source

# Optional development inspection — uncomment only when needed.
# display(df)
# display(profile_result["profile"])
# display(dq_df)
# display(dq_failed_values)

# 3. Transform

Use the named source DataFrames to build the DataFrames you want to publish. This example produces two different outputs so the WRITE blocks demonstrate publishing different tables.

In [ ]:
# Pull the source DataFrames back out of the sources dictionary by name.
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

# Output 1: detailed order-level table.
transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn("order_net_amount", F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2))
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

# Output 2: customer-level summary derived from the detailed order output.
customer_summary_df = (
    transformed_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("order_net_amount").alias("total_order_net_amount"),
        F.max("order_datetime").alias("latest_order_datetime"),
        F.max("historical_order_count").alias("historical_order_count"),
        F.max("historical_net_amount").alias("historical_net_amount"),
    )
)

# Optional development inspection — uncomment only when needed.
# display(transformed_df)
# display(customer_summary_df)

# 4. Write

Each WRITE block is one complete target flow: define the target, run its Guardrails, confirm contract coverage, publish it, then profile the persisted table. Clone a WRITE block for every additional target.

In [ ]:
# Dictionary used to keep multiple write results for later reference.
# Key = WRITE_NAME (for example "curated_orders_lakehouse").
# Value = the pipeline_write() result, including the written target table_id.
writes = {}

## WRITE 1 — Lakehouse Curated Orders

In [ ]:
WRITE_NAME = "curated_orders_lakehouse"
WRITE_DATAFRAME = transformed_df
WRITE_SOURCE_NAMES = ("orders", "products", "history")
WRITE_STORE = "Silver"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "curated_orders"
WRITE_LOAD_STRATEGY = "overwrite"

# Optional Spark write parallelism.
# Check spark.sparkContext.defaultParallelism to see the session's available task parallelism.
# Example: 64 allows up to 64 write tasks, but only as many as the current Spark capacity can run concurrently.
WRITE_REPARTITION_BY = None

# Pick only the source reads that actually feed this target.
# FabricOps uses their table_id values for Source Drift, coverage and Lineage.
write_sources = [sources[name] for name in WRITE_SOURCE_NAMES]

# Resolve the target once so every pre-write check uses the same canonical table_id.
target_table_id = resolve_table_id(store=WRITE_STORE, schema=WRITE_SCHEMA, table_name=WRITE_TABLE)

# Stop here if the output columns or data types do not match the target contract.
target_schema_result = check_schema(WRITE_DATAFRAME, table_id=target_table_id, raise_on_failure=True)

# Apply required masking, redaction, hashing, or tokenization before writing.
sensitive_result = check_sensitive_data(WRITE_DATAFRAME, table_id=target_table_id, raise_on_failure=True)
prepared_df = sensitive_result["dataframe"]
support_mapping_df = sensitive_result.get("support_mapping")

# Compare each source with the last version successfully used by this target.
for source in write_sources:
    check_source_drift(source["table_id"], target_table_id=target_table_id, raise_on_failure=True)

# Run the target DQ rules and keep the checked DataFrame and failed rows for optional use.
target_dq_result = check_dq(prepared_df, table_id=target_table_id, raise_on_failure=True)
target_dq_df = target_dq_result.get("dataframe", prepared_df)
target_dq_failed_values = target_dq_result.get("failed_values")

# Confirm every configured Guardrail that applies to this publication actually ran.
# Missing coverage warns in Development and blocks before publication in Production.
coverage_result = check_guardrail_coverage(target_table_id=target_table_id, source_table_ids=[source["table_id"] for source in write_sources])

# Optional development inspection — uncomment only when needed.
# display(prepared_df)
# display(target_dq_df)
# display(target_dq_failed_values)
# display(support_mapping_df)

# Publish the governed target by the same canonical table_id used for the pre-write checks.
write_result = pipeline_write(prepared_df, table_id=target_table_id, load_strategy=WRITE_LOAD_STRATEGY, source_table_ids=[source["table_id"] for source in write_sources], repartition_by=WRITE_REPARTITION_BY)

# Save this write result by name so later cells can reuse the target table_id.
writes[WRITE_NAME] = write_result

# Refresh the profile from the table that was actually written.
write_profile = profile_table(table_id=write_result["table_id"])

# Optional development inspection — uncomment only when needed.
# display(write_profile["profile"])

## WRITE 2 — Warehouse Customer Summary

In [ ]:
WRITE_NAME = "customer_summary_warehouse"
WRITE_DATAFRAME = customer_summary_df
WRITE_SOURCE_NAMES = ("orders", "products", "history")
WRITE_STORE = "Gold"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "customer_summary"
WRITE_LOAD_STRATEGY = "overwrite"

# Optional Spark write parallelism.
# Check spark.sparkContext.defaultParallelism to see the session's available task parallelism.
# Example: 64 allows up to 64 write tasks, but only as many as the current Spark capacity can run concurrently.
WRITE_REPARTITION_BY = None

# Pick only the source reads that actually feed this target.
# FabricOps uses their table_id values for Source Drift, coverage and Lineage.
write_sources = [sources[name] for name in WRITE_SOURCE_NAMES]

# Resolve the target once so every pre-write check uses the same canonical table_id.
target_table_id = resolve_table_id(store=WRITE_STORE, schema=WRITE_SCHEMA, table_name=WRITE_TABLE)

# Stop here if the output columns or data types do not match the target contract.
target_schema_result = check_schema(WRITE_DATAFRAME, table_id=target_table_id, raise_on_failure=True)

# Apply required masking, redaction, hashing, or tokenization before writing.
sensitive_result = check_sensitive_data(WRITE_DATAFRAME, table_id=target_table_id, raise_on_failure=True)
prepared_df = sensitive_result["dataframe"]
support_mapping_df = sensitive_result.get("support_mapping")

# Compare each source with the last version successfully used by this target.
for source in write_sources:
    check_source_drift(source["table_id"], target_table_id=target_table_id, raise_on_failure=True)

# Run the target DQ rules and keep the checked DataFrame and failed rows for optional use.
target_dq_result = check_dq(prepared_df, table_id=target_table_id, raise_on_failure=True)
target_dq_df = target_dq_result.get("dataframe", prepared_df)
target_dq_failed_values = target_dq_result.get("failed_values")

# Confirm every configured Guardrail that applies to this publication actually ran.
# Missing coverage warns in Development and blocks before publication in Production.
coverage_result = check_guardrail_coverage(target_table_id=target_table_id, source_table_ids=[source["table_id"] for source in write_sources])

# Optional development inspection — uncomment only when needed.
# display(prepared_df)
# display(target_dq_df)
# display(target_dq_failed_values)
# display(support_mapping_df)

# Publish the governed target by the same canonical table_id used for the pre-write checks.
write_result = pipeline_write(prepared_df, table_id=target_table_id, load_strategy=WRITE_LOAD_STRATEGY, source_table_ids=[source["table_id"] for source in write_sources], repartition_by=WRITE_REPARTITION_BY)

# Save this write result by name so later cells can reuse the target table_id.
writes[WRITE_NAME] = write_result

# Refresh the profile from the table that was actually written.
write_profile = profile_table(table_id=write_result["table_id"])

# Optional development inspection — uncomment only when needed.
# display(write_profile["profile"])